In [ ]:
# Python (run once at top of notebook)
from pathlib import Path
import sys
import json
import pandas as pd

# assume notebook is in project_root/notebooks
PROJECT_ROOT = Path.cwd().resolve().parent  # adjust if you run notebook from elsewhere
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# then import
from upwork_scraper.connectors import supabase

In [ ]:
supa_client = supabase.supabase

response = supa_client.table('jobs').select("*", count="exact").limit(100).execute()
sample_df = pd.DataFrame(response.data)

sample_df.to_json("sample.jsonl", orient="records",lines=True)

In [41]:
import os
from typing import List, Dict, Optional, Any


sb = supabase.supabase


def _check_response(resp: Any):
    # Generic response checker for different supabase-py versions
    if resp is None:
        raise RuntimeError("No response received from Supabase client.")
    # If it's a dict-like
    if isinstance(resp, dict):
        if resp.get("error"):
            raise RuntimeError(f"Supabase error: {resp['error']}")
        return resp.get("data")
    # If it has status_code and data attributes (common)
    data = getattr(resp, "data", None)
    status = getattr(resp, "status_code", None)
    error = getattr(resp, "error", None)
    if error:
        # error may be an object with message
        msg = error.message if hasattr(error, "message") else str(error)
        raise RuntimeError(f"Supabase error: {msg}")
    if status is not None and status >= 400:
        raise RuntimeError(f"Supabase returned status {status}: {data}")
    return data


def _extract_query_from_expansion(expanded: Any) -> Optional[str]:
    # expansion can be dict, list, or None
    if expanded is None:
        return None
    if isinstance(expanded, list):
        if not expanded:
            return None
        # take first element
        elem = expanded[0]
        return elem.get("query") if isinstance(elem, dict) else None
    if isinstance(expanded, dict):
        return expanded.get("query")
    return None


def distinct_queries_per_job() -> List[Dict]:
    """
    Returns list of {"job_id": ..., "query": ...} for distinct (job_id, query).
    Uses a single request expanding scrape_requests from search_results.
    """
    resp = sb.table("search_results").select("job_id, scrape_requests(query)").execute()
    rows = _check_response(resp)
    if not rows:
        return []

    results = []
    seen = set()
    for r in rows:
        job_id = r.get("job_id")
        expanded = r.get("scrape_requests")
        query = _extract_query_from_expansion(expanded)
        if query is None:
            continue
        key = (job_id, query)
        if key in seen:
            continue
        seen.add(key)
        results.append({"job_id": job_id, "query": query})
    return results


def latest_query_per_job_via_view() -> List[Dict]:
    """
    Recommended: Create a DB view that returns one row per job with the latest query by query_timestamp,
    then call it via Supabase in a single request.

    SQL for view (run once via SQL editor or execute_sql):
    ---------------------------------------------------
    CREATE VIEW public.job_latest_query AS
    SELECT
      j.job_id,
      sr.query
    FROM public.jobs j
    LEFT JOIN LATERAL (
      SELECT sr2.query
      FROM public.search_results s2
      JOIN public.scrape_requests sr2 ON sr2.search_id = s2.search_id
      WHERE s2.job_id = j.job_id AND sr2.query IS NOT NULL
      ORDER BY sr2.query_timestamp DESC NULLS LAST
      LIMIT 1
    ) sr ON true;
    ---------------------------------------------------

    Then call the view:
    """
    resp = sb.table("job_latest_query").select("*").execute()
    rows = _check_response(resp)
    if not rows:
        return []
    return [{"job_id": r.get("job_id"), "query": r.get("query")} for r in rows]


def latest_query_per_job_n_plus_one() -> List[Dict]:
    """
    Fallback that does one request per job (N+1). Use only for small job sets.
    """
    resp = sb.table("jobs").select("job_id").execute()
    jobs = _check_response(resp)
    if not jobs:
        return []

    results = []
    for j in jobs:
        job_id = j.get("job_id")
        # Expand scrape_requests via search_results, order by query_timestamp desc limit 1
        resp = sb.table("search_results") \
                 .select("scrape_requests(query,query_timestamp)") \
                 .eq("job_id", job_id) \
                 .order("scrape_requests.query_timestamp", desc=True) \
                 .limit(1) \
                 .execute()
        rows = _check_response(resp)
        query_val = None
        if rows:
            expanded = rows[0].get("scrape_requests")
            query_val = _extract_query_from_expansion(expanded)
        results.append({"job_id": job_id, "query": query_val})
    return results

print("Distinct queries per job (sample):")
try:
    dq = distinct_queries_per_job()
    for i, row in enumerate(dq[:20]):
        print(i, row)
except Exception as e:
    print("Error:", e)

print("\nLatest query per job via view (recommended):")
try:
    lq = latest_query_per_job_via_view()
    for i, row in enumerate(lq[:20]):
        print(i, row)
except Exception as e:
    print("View not found or error:", e)
    print("Falling back to N+1 approach...")
    try:
        lq2 = latest_query_per_job_n_plus_one()
        for i, row in enumerate(lq2[:20]):
            print(i, row)
    except Exception as e2:
        print("Error in N+1 approach:", e2)


Distinct queries per job (sample):
0 {'job_id': '1939352305698018414', 'query': 'data scientist python'}
1 {'job_id': '1939368833898877971', 'query': 'data scientist python'}
2 {'job_id': '1939370324537742355', 'query': 'data scientist python'}
3 {'job_id': '1939374443817538670', 'query': 'data scientist python'}
4 {'job_id': '1939377628973495207', 'query': 'data scientist python'}
5 {'job_id': '1939386663674778734', 'query': 'data scientist python'}
6 {'job_id': '1939389242555864083', 'query': 'data scientist python'}
7 {'job_id': '1939416916137448467', 'query': 'data scientist python'}
8 {'job_id': '1939448314341204401', 'query': 'data scientist python'}
9 {'job_id': '1939449142758470567', 'query': 'data scientist python'}
10 {'job_id': '1939479387404310668', 'query': 'data scientist python'}
11 {'job_id': '1939479495536027693', 'query': 'data scientist python'}
12 {'job_id': '1939486370278964964', 'query': 'data scientist python'}
13 {'job_id': '1939507694016762899', 'query': 'data 

In [39]:
# --- Upwork Feature Engineering Pipeline (JSONL -> Engineered CSV/DataFrame) ---

import json
from datetime import timezone
import numpy as np
import pandas as pd

# --------- INPUTS ---------
JSONL_PATH = "/mnt/data/sample.jsonl"   # <- change as needed
OUTPUT_CSV = "/mnt/data/engineered_features.csv"  # <- change as needed

# Optional: if you already have a mapping DataFrame (job_id, search_term),
# set MAPPING_DF to that DataFrame before running the "JOIN (optional)" block.
MAPPING_DF = None  # placeholder; see example near the bottom

# --------- LOAD ---------
# records = []
# with open(JSONL_PATH, "r", encoding="utf-8") as f:
#     for line in f:
#         try:
#             records.append(json.loads(line))
#         except json.JSONDecodeError:
#             # Skip lines that aren't valid JSON
#             continue

df = sample_df

# Ensure expected columns exist (prevents KeyErrors if a field is missing)
expected_cols = [
    "job_id","url","title","description","skills","created_on","published_on","duration_label",
    "job_type","tier_text","fixed_budget","weekly_budget","hourly_budget_min","hourly_budget_max",
    "currency","client_country","client_total_spent","client_payment_verified","client_total_reviews",
    "client_avg_feedback","proposals_tier"
]
for c in expected_cols:
    if c not in df.columns:
        df[c] = np.nan

# --------- HELPERS ---------
def parse_dt(x):
    if pd.isna(x): return pd.NaT
    try:
        return pd.to_datetime(x, utc=True)
    except Exception:
        return pd.NaT

def proposals_bounds(tier):
    """
    Parse 'proposals_tier' strings like "5to10", "20to50", "50plus" into (min, max).
    For "Xplus", uses (X, X+50) as a broad upper bound placeholder.
    """
    if pd.isna(tier): return (np.nan, np.nan)
    s = str(tier).lower()
    if "plus" in s:
        try:
            val = int(s.replace("plus", ""))
            return (val, val + 50)
        except Exception:
            return (np.nan, np.nan)
    if "to" in s:
        parts = s.split("to")
        try:
            lo, hi = int(parts[0]), int(parts[1])
            return (lo, hi)
        except Exception:
            return (np.nan, np.nan)
    try:
        val = int(s)
        return (val, val)
    except Exception:
        return (np.nan, np.nan)

def duration_weeks_map(label):
    """
    Rough mapping from common Upwork duration labels to weeks.
    Tweak as needed to match your labels.
    """
    if pd.isna(label): return np.nan
    s = str(label).lower().strip()
    if "less than 1 month" in s: return 4
    if "1 to 3 months" in s or "1–3 months" in s or "1-3 months" in s: return 10
    if "3 to 6 months" in s or "3–6 months" in s or "3-6 months" in s: return 20
    if "more than 6 months" in s or "6+" in s: return 28
    if "week" in s and any(str(i) in s for i in range(1, 9)):
        # Extract first int as weeks
        try:
            n = int("".join(ch for ch in s if ch.isdigit()))
            return n
        except Exception:
            return np.nan
    return np.nan

def safe_len_words(x):
    if isinstance(x, str): return len(x.split())
    return 0

def hourly_median(lo, hi):
    """
    Median of hourly min/max with NaN handling; if only one bound exists, use it.
    """
    try:
        lo = float(lo) if pd.notna(lo) else np.nan
        hi = float(hi) if pd.notna(hi) else np.nan
    except Exception:
        return np.nan
    if pd.notna(lo) and pd.notna(hi): return (lo + hi) / 2
    if pd.notna(lo): return lo
    if pd.notna(hi): return hi
    return np.nan

def unified_value(row, assumed_hours=20):
    """
    Unifies fixed + hourly into a comparable "value magnitude".
    For hourly: hourly_median * assumed_hours (default 20).
    """
    jt = str(row.get("job_type", "")).lower() if pd.notna(row.get("job_type")) else ""
    if jt == "fixed":
        fb = row.get("fixed_budget", np.nan)
        try:
            fb = float(fb)
        except Exception:
            fb = np.nan
        return fb if pd.notna(fb) else np.nan
    # hourly or unknown
    hr = row.get("hourly_rate_median", np.nan)
    try:
        hr = float(hr)
    except Exception:
        hr = np.nan
    return hr * assumed_hours if pd.notna(hr) else np.nan

def spend_bucket(spent):
    try:
        s = float(spent)
    except Exception:
        return np.nan
    if s < 1_000: return "<1k"
    if s < 10_000: return "1–10k"
    if s < 50_000: return "10–50k"
    return ">50k"

def norm01(s: pd.Series) -> pd.Series:
    """
    Min-max normalize to [0,1]; robust to NaNs/constant series.
    """
    s = s.astype(float)
    if s.dropna().empty:  # all NaN
        return pd.Series(np.zeros(len(s)), index=s.index)
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx == mn:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

# --------- FEATURE ENGINEERING ---------
# Demand
df["created_dt"]        = df["created_on"].apply(parse_dt)
now_utc                 = pd.Timestamp.now(tz=timezone.utc)
df["days_since_posted"] = (now_utc - df["created_dt"]).dt.days
df["description_length"]= df["description"].apply(safe_len_words)
df["skills_count"]      = df["skills"].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Value
df["hourly_rate_median"]= df.apply(lambda r: hourly_median(r["hourly_budget_min"], r["hourly_budget_max"]), axis=1)
df["duration_weeks_est"]= df["duration_label"].apply(duration_weeks_map)
df["pricing_type"]      = df["job_type"].str.lower().fillna("unknown")
df["budget_value"]      = df.apply(unified_value, axis=1)

# Client
df["client_payment_verified_int"] = df["client_payment_verified"].astype(float).replace({True: 1.0, False: 0.0})
df["client_spend_category"]       = df["client_total_spent"].apply(spend_bucket)

# Competition
bounds = df["proposals_tier"].apply(proposals_bounds).apply(pd.Series)
df["proposals_min"]      = bounds[0]
df["proposals_max"]      = bounds[1]
df["proposals_midpoint"] = (df["proposals_min"] + df["proposals_max"]) / 2.0

# Experience level (cleaned)
df["experience_level"] = df["tier_text"].astype(str).str.title()

# Placeholder for search term (filled after join)
df["search_term"] = pd.NA

# Client Quality Index (CQI)
cq_spend    = norm01(df["client_total_spent"])
cq_feedback = norm01(df["client_avg_feedback"])
cq_verified = df["client_payment_verified_int"].fillna(0.0)
cq_reviews  = norm01(df["client_total_reviews"])

df["client_quality_index"] = (
    0.4 * cq_spend +
    0.3 * cq_feedback +
    0.2 * cq_verified +
    0.1 * cq_reviews
)

# Competition z-score of proposals_midpoint (relative)
prop = df["proposals_midpoint"].astype(float)
mu   = prop.mean(skipna=True)
sd   = prop.std(skipna=True, ddof=0)
df["competition_zscore"] = (prop - mu) / sd if (isinstance(sd, float) and np.isfinite(sd) and sd > 0) else 0.0


import re

TECH_TERMS = ["python", "sql", "excel", "powerbi", "tableau", "supabase", "api", "gcp", "aws", "fastapi"]
AI_TERMS = ["ai", "machine learning", "ml", "gpt", "llm", "chatgpt", "genai"]
REPORT_TERMS = ["dashboard", "report", "reporting", "bi", "visualization"]
FORECAST_TERMS = ["forecast", "predict", "model", "optimize", "optimization"]

def count_terms(text, terms):
    if not isinstance(text, str): return 0
    text = text.lower()
    return sum(term in text for term in terms)

def has_any(text, terms):
    return count_terms(text, terms) > 0

df["title_len"] = df["title"].apply(lambda t: len(str(t).split()))
df["description_len"] = df["description"].apply(lambda d: len(str(d).split()))
df["tech_terms_count"] = df["description"].apply(lambda t: count_terms(t, TECH_TERMS))
df["mentions_ai"] = df["description"].apply(lambda t: has_any(t, AI_TERMS))
df["mentions_reporting"] = df["description"].apply(lambda t: has_any(t, REPORT_TERMS))
df["mentions_forecast"] = df["description"].apply(lambda t: has_any(t, FORECAST_TERMS))
df["question_count"] = df["description"].str.count(r"\?")
df["bullet_count"] = df["description"].str.count(r"[\n•\-]\s")
df["urgency_flag"] = df["description"].str.contains(r"\b(asap|urgent|immediately|today)\b", case=False, na=False)
df["seniority_flag"] = df["description"].str.contains(r"\b(senior|expert|specialist)\b", case=False, na=False)

# --------- COLUMN ORDER / EXPORT ---------
base_cols = [
    # --- Identification & Metadata ---
    "job_id", "url", "title", "pricing_type", "currency", "experience_level",
    "duration_label", "duration_weeks_est",

    # --- Value Features ---
    "hourly_budget_min", "hourly_budget_max", "hourly_rate_median",
    "fixed_budget", "budget_value",

    # --- Client Quality ---
    "client_country", "client_payment_verified", "client_total_spent",
    "client_spend_category", "client_total_reviews", "client_avg_feedback",
    "client_quality_index",

    # --- Competition ---
    "proposals_tier", "proposals_min", "proposals_max", "proposals_midpoint",
    "competition_zscore",

    # --- Temporal & Demand ---
    "created_on", "days_since_posted", "description_length", "skills_count",

    # --- Text-Derived Features ---
    "title_len",                # number of words in title
    "description_len",          # number of words in description
    "tech_terms_count",         # number of matched tech stack keywords
    "mentions_ai",              # flag for AI/ML-related terms
    "mentions_reporting",       # flag for BI/reporting-related terms
    "mentions_forecast",        # flag for forecasting/modeling terms
    "question_count",           # number of '?' characters (client uncertainty proxy)
    "bullet_count",             # number of bullet/structured list items
    "urgency_flag",             # boolean flag for urgency mentions
    "seniority_flag",           # boolean flag for seniority/expertise language

    # --- Contextual ---
    "skills", "search_term"     # retained for later join and skill references
]
export_cols = [c for c in base_cols if c in df.columns]
engineered = df[export_cols].copy()

# Write CSV (Parquet omitted to avoid engine deps)
# engineered.to_csv(OUTPUT_CSV, index=False)
# print(f"Engineered features saved to: {OUTPUT_CSV}")

# --------- JOIN (optional): add search_term via your mapping table ---------
# If you have a mapping DataFrame with columns ["job_id", "search_term"], uncomment below:
#
# if MAPPING_DF is not None:
#     # Ensure dtypes and de-dup
#     map_df = (MAPPING_DF[["job_id", "search_term"]]
#               .dropna(subset=["job_id", "search_term"])
#               .drop_duplicates())
#     # If a job can come from multiple searches, you can keep first, or aggregate:
#     # Option A (keep first):
#     # map_df = map_df.drop_duplicates(subset=["job_id"], keep="first")
#     #
#     # Option B (aggregate multiple terms into a single string):
#     # map_df = map_df.groupby("job_id")["search_term"].apply(lambda s: "; ".join(sorted(set(s)))).reset_index()
#     #
#     engineered = engineered.merge(map_df, on="job_id", how="left", suffixes=("", "_mapped"))
#     # Prefer mapped search_term if present
#     engineered["search_term"] = engineered["search_term_mapped"].combine_first(engineered["search_term"])
#     engineered = engineered.drop(columns=[c for c in ["search_term_mapped"] if c in engineered.columns])
#     # (Optional) re-save with search terms
#     # engineered.to_csv(OUTPUT_CSV.replace(".csv", "_with_terms.csv"), index=False)

# --------- (OPTIONAL) NICHE ROLLUPS once search_term is populated ---------
# Example aggregations to compute per-keyword metrics:
#
# if "search_term" in engineered.columns and engineered["search_term"].notna().any():
#     g = engineered.dropna(subset=["search_term"]).groupby("search_term", dropna=True)
#     rollup = pd.DataFrame({
#         "jobs": g.size(),
#         "median_hourly": g["hourly_rate_median"].median(),
#         "median_value": g["budget_value"].median(),
#         "median_proposals": g["proposals_midpoint"].median(),
#         "mean_client_quality": g["client_quality_index"].mean(),
#         "p_verified_clients": g["client_payment_verified"].mean(),
#     }).reset_index()
#
#     # Composite indices (example):
#     def norm01_col(x): 
#         x = x.astype(float)
#         return (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0.0
#     rollup["MDI"] = norm01_col(rollup["jobs"] * 1.0)  # Market Demand Index
#     rollup["VPI"] = norm01_col(rollup["median_hourly"].fillna(0) * rollup["median_value"].fillna(0))  # Value Potential
#     rollup["CAO"] = (rollup["MDI"] * rollup["VPI"]) / (1 + rollup["median_proposals"].fillna(0))       # Competition-Adjusted Opportunity
#     rollup = rollup.sort_values(["CAO","VPI","MDI"], ascending=False)
#     display(rollup.head(25))

# Preview (optional): display the first 10 engineered rows in the notebook
engineered.head(10)

/var/folders/hy/pml03tfn6dxfd7508zw3bsxm0000gn/T/ipykernel_22539/3096067926.py:227: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["urgency_flag"] = df["description"].str.contains(r"\b(asap|urgent|immediately|today)\b", case=False, na=False)
/var/folders/hy/pml03tfn6dxfd7508zw3bsxm0000gn/T/ipykernel_22539/3096067926.py:228: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["seniority_flag"] = df["description"].str.contains(r"\b(senior|expert|specialist)\b", case=False, na=False)


,job_id,url,title,pricing_type,currency,experience_level,duration_label,duration_weeks_est,hourly_budget_min,hourly_budget_max,...,tech_terms_count,mentions_ai,mentions_reporting,mentions_forecast,question_count,bullet_count,urgency_flag,seniority_flag,skills,search_term
0,1987969201035894058,https://www.upwork.com/jobs/~02198796920103589...,Data analyst Interview Preparation Specialist,fixed,USD,Expert,Less than 1 month,4,0,0.0,...,3,False,False,False,0,0,False,False,"[Python, SQL, R, Snowflake, PostgreSQL Program...",<NA>
1,1987965747742336139,https://www.upwork.com/jobs/~02198796574774233...,Freelance AI Software Developer (AI SEO & Cont...,hourly,USD,Expert,More than 6 months,28,30,50.0,...,5,True,True,True,0,46,False,False,"[Search Engine Optimization, WordPress, Web De...",<NA>
2,1987962662485100395,https://www.upwork.com/jobs/~02198796266248510...,Data Scientist for Product Recommendation Engine,hourly,USD,Expert,Less than 1 month,4,30,60.0,...,0,True,True,False,0,0,False,False,"[Python, Data Science, Python Scikit-Learn, LL...",<NA>
3,1987932535604774763,https://www.upwork.com/jobs/~02198793253560477...,Python Developer / Data Engineer to Build Inte...,fixed,USD,Expert,Less than 1 month,4,0,0.0,...,1,False,False,False,0,0,False,False,"[Data Mining, JavaScript, Web Development]",<NA>
4,1987924994803981610,https://www.upwork.com/jobs/~02198792499480398...,Financial Document Parameter Extraction Specia...,hourly,USD,Intermediate,1 to 3 months,10,0,0.0,...,0,True,True,True,0,0,False,False,"[Data Entry, Python, Data Scraping, Microsoft ...",<NA>
5,1987924323417231665,https://www.upwork.com/jobs/~02198792432341723...,AI and Machine Learning Coding Tutorial Creator,fixed,USD,Intermediate,Less than 1 month,4,0,0.0,...,1,True,False,False,0,0,False,False,"[Python, Amazon SageMaker, JavaScript, LangCha...",<NA>
6,1987904024148008234,https://www.upwork.com/jobs/~02198790402414800...,Experienced AI and LLM Engineer - FOR RAG,fixed,USD,Intermediate,1 to 3 months,10,0,0.0,...,3,True,False,False,0,16,False,False,"[Python, Data Preprocessing, AI App Developmen...",<NA>
7,1987883084521495850,https://www.upwork.com/jobs/~02198788308452149...,AI-Focused Software Engineer Needed for Projec...,hourly,USD,Intermediate,More than 6 months,28,3,20.0,...,0,True,False,False,0,0,False,False,"[Python, Artificial Intelligence, Machine Lear...",<NA>
8,1987839492411186474,https://www.upwork.com/jobs/~02198783949241118...,AI Developer for LLM-Based Data Analysis on Am...,hourly,USD,Expert,1 to 3 months,10,15,50.0,...,1,True,False,True,0,3,False,False,"[Data Analysis, AI Builder, AI Agent Developme...",<NA>
9,1987829010417715051,https://www.upwork.com/jobs/~02198782901041771...,Developer with Expertise in Differential Geome...,fixed,USD,Expert,1 to 3 months,10,0,0.0,...,1,False,False,False,0,0,False,False,"[Python, PyTorch]",<NA>


In [43]:
import yaml
file_path = r"/Users/jslade/Documents/GitHub/upwork_scraper/config/search_urls.yml"
with open(file_path, 'r') as f:
    data = yaml.safe_load(f)
    urls = data.get('urls', {}).get('section0',[])

print(urls)

['https://www.upwork.com/nx/search/jobs/?payment_verified=1&per_page=50&sort=recency&q=data%20scientist%20python&page=1', 'https://www.upwork.com/nx/search/jobs/?payment_verified=1&per_page=50&sort=recency&q=data%20scientist%20python&page=2', 'https://www.upwork.com/nx/search/jobs/?payment_verified=1&per_page=50&sort=recency&q=data%20scientist%20python&page=3']
